In [1]:
import numpy as np

from scipy.fft import fft, ifft, dct, idct, dst, idst

from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation, QFTGate, UnitaryGate
from qiskit.quantum_info import Statevector

In [2]:
WINDOW_SIZES = [256] #2^8 must n^2: 64, 128, 256, 512

def validate_window(x):
    x = np.asarray(x)
    if x.ndim != 1:
        raise ValueError("Transform input must be 1-D")

    n = len(x)
    if n == 0 or (n & (n-1)) != 0:
        raise ValueError(f"Window length must be a power of two, got {n}")

    return x 

# Classical FFT (Fast Fourier Transforms)

In [3]:
def classical_fft(x):
    x = validate_window(x)
    return fft(x, norm="ortho")

def classical_ifft(coeffs):
    coeffs = validate_window(coeffs)
    return ifft(coeffs, norm="ortho")

# Classical DCT-II (Discrete Cosine Transforms II)

In [4]:
def classical_dct(x):
    x = validate_window(x)
    return dct(x, type=2, norm="ortho")

def classical_idct(coeffs):
    coeffs = validate_window(coeffs)
    return idct(coeffs, type=2, norm="ortho")

# Classical DST-II (Discrete Sine Transforms II)

In [5]:
def classical_dst(x):
    x = validate_window(x)
    return dst(x, type=2, norm="ortho")


def classical_idst(coeffs):
    coeffs = validate_window(coeffs)
    return idst(coeffs, type=2, norm="ortho")

# Classical DWT-Haar (Discrete Wavelet Transforms Haar)

In [6]:
def classical_haar(x):
    x = validate_window(x).astype(np.float64)
    approx = x.copy()
    details = []
    while len(approx) > 1:
        even = approx[0::2]
        odd = approx[1::2]
        next_approx = (even + odd) / np.sqrt(2.0)
        detail = (even - odd) / np.sqrt(2.0)
        details.append(detail)
        approx = next_approx

    return np.concatenate([approx, *details[::-1]])

def classical_ihaar(coeffs):
    coeffs = validate_window(coeffs).astype(np.float64)
    approx = coeffs[:1]
    offset = 1
    while offset < len(coeffs):
        n = len(approx)
        detail = coeffs[offset:offset+n]
        reconstructed = np.empty(2 * n, dtype=np.float64)
        reconstructed[0::2] = (approx + detail) / np.sqrt(2.0)
        reconstructed[1::2] = (approx - detail) / np.sqrt(2.0)
        approx = reconstructed
        offset += n

    return approx

x = np.random.default_rng(42).normal(size=256)
X = classical_haar(x)
x_hat = classical_ihaar(X)
print(np.max(np.abs(x - x_hat)))

1.3322676295501878e-15


# Quantum encodings

In [7]:
def prepare_amplitudes(x):
    x = validate_window(x)
    norm = np.linalg.norm(x)
    if norm <= 1e-15:
        return None, 0.0

    amplitudes = (x / norm).astype(np.complex128)
    num_qubits = int(np.log2(len(x)))
    qc = QuantumCircuit(num_qubits)
    qc.append(StatePreparation(amplitudes), range(num_qubits),)
    return qc, norm

# QFT

In [8]:
def quantum_fft(x):
    qc, input_norm = prepare_amplitudes(x)

    if qc is None:
        return np.zeros_like(x, dtype=np.complex128), None

    num_qubits = qc.num_qubits
    qc.append(QFTGate(num_qubits).inverse(), range(num_qubits))
    state = Statevector.from_instruction(qc).data
    coeffs = (np.asarray(state) * input_norm)
    return coeffs, qc

def quantum_ifft(coeffs):
    qc, coeff_norm = prepare_amplitudes(coeffs)

    if qc is None:
        return np.zeros_like(coeffs, dtype=np.complex128), None

    num_qubits = qc.num_qubits
    qc.append(QFTGate(num_qubits), range(num_qubits))
    state = Statevector.from_instruction(qc).data
    reconstructed = (np.asarray(state) * coeff_norm)
    return reconstructed, qc

X_c = classical_fft(x)
X_q, qc = quantum_fft(x)

print(np.max(np.abs(X_c - X_q)))

2.1935456002220668e-13


# DCT Matrixes

In [9]:
def dct_matrix(n):
    eye = np.eye(n)
    return dct(eye, type=2, norm="ortho", axis=0)

U = dct_matrix(256)

print(np.max(np.abs(U.T @ U - np.eye(256))))

1.2212453270876722e-15


# Quantum Matrixes Transform

In [10]:
def quantum_unitary_transform(x, matrix, label):
    qc, input_norm = prepare_amplitudes(x)

    if qc is None:
        return np.zeros_like(x, dtype=np.complex128), None

    gate = UnitaryGate(matrix, label=label,)
    qc.append(gate, range(qc.num_qubits),)
    state = Statevector.from_instruction(qc).data
    coeffs = (np.asarray(state) * input_norm)
    return coeffs, qc

def quantum_unitary_inverse(coeffs, matrix, label):
    return quantum_unitary_transform(coeffs, matrix.conj().T, label)

# QDCT-II (Quantum Discete Cosine Transform II)

In [11]:
def quantum_dct(x):
    x = validate_window(x)
    U = dct_matrix(len(x))
    return quantum_unitary_transform(x, U, "QDCT-II")

def quantum_idct(coeffs):
    coeffs = validate_window(coeffs)
    U = dct_matrix(len(coeffs))
    return quantum_unitary_inverse(coeffs, U, "IQDCT")

X_c = classical_dct(x)
X_q, _ = quantum_dct(x)

print(np.max(np.abs(X_c - X_q.real)))

2.6711965972481266e-13
